In [15]:
import pandas as pd
from clickhouse_connect import get_client
def comma(index,num_queries):
    """
    Возвращает запятую и перенос строки, если индекс меньше общего количества запросов,
    иначе возвращает пустую строку.
    """
    if index < num_queries: 
        return ', \n'
    else:
        return ''

def generate_metric_queries(metric_dict):
    """
    Генерирует строки запросов для каждой категории метрик из словаря metric_dict.
    Возвращает словарь с запросами для каждой категории.
    """
    def build_query_string(metrics):
        return ', \n'.join(metrics.values())

    return {
        category: build_query_string(metrics)
        for category, metrics in metric_dict.items()
    }

def make_query(metric_dict, config):
    """
    Создает полный SQL-запрос на основе словаря метрик и конфигурации.
    Включает подзапросы для заказов, сессий, аутентификации и продуктов.
    """
    metric_queries = generate_metric_queries(metric_dict)

    order_metric_queries = metric_queries['order']
    session_metric_queries = metric_queries['session']
    auth_metric_queries = metric_queries['auth']
    product_metric_queries = metric_queries['product']
        
    order_query = f"""
        SELECT
            CONCAT('{config['start_date']}', ' - ', '{config['end_date']}') as interval,
            {order_metric_queries}
        FROM digital_product_analytics.app_metric_events AS t1
        WHERE t1.event_name IN ('system_order_create_success') AND 
        toDate(t1.event_datetime) >= '{config['start_date']}' AND toDate(t1.event_datetime) < '{config['end_date']}'
        LIMIT 1000001
    """
    session_query = f"""
        SELECT
            CONCAT('{config['start_date']}', ' - ', '{config['end_date']}') as interval,
            {session_metric_queries}
        FROM digital_product_analytics.app_metric_events AS t1
        WHERE toDate(t1.event_datetime) >= '{config['start_date']}' AND toDate(t1.event_datetime) < '{config['end_date']}'
        LIMIT 1000001
    """
    auth_query = f"""
        SELECT
            CONCAT('{config['start_date']}', ' - ', '{config['end_date']}') as interval,
            {auth_metric_queries}
        FROM digital_product_analytics.app_metric_events AS t1
        WHERE t1.event_name IN ('system_authentication_showed', 'system_order_create_success', 'system_authentication_success') 
        AND toDate(t1.event_datetime) >= '{config['start_date']}' AND toDate(t1.event_datetime) < '{config['end_date']}'
        AND t1.kiosk_session_id NOT IN ('')
        LIMIT 1000001
    """
    product_query = f"""
        SELECT
            CONCAT('{config['start_date']}', ' - ', '{config['end_date']}') as interval,
            {product_metric_queries}
        FROM digital_product_analytics.app_metric_events AS t1
        WHERE t1.event_name IN ('user_product_added')
        AND toDate(t1.event_datetime) >= '{config['start_date']}' AND toDate(t1.event_datetime) < '{config['end_date']}'
        LIMIT 1000001
    """

    query = f"""
        SELECT 
        order.interval as interval,
        *
        FROM 
        ({order_query}) as order
        JOIN ({session_query}) as session ON order.interval = session.interval
        JOIN ({auth_query}) as auth ON order.interval = auth.interval
        JOIN ({product_query}) as product ON order.interval = product.interval 
    """

    print(query)
    return query

def get_stat_query(stat_dict, config1, config2, method_dict):
    """
    Формирует SQL-запрос для статистического анализа на основе словарей статистик и методов,
    а также двух конфигураций периодов.
    """
    def build_query_string(metrics):
        return ', '.join(metrics.values())

    metric_queries = []
    method_queries = []

    metric_queries.append(build_query_string(stat_dict))
    method_queries.append(build_query_string(method_dict))

    metric_queries = ', '.join(metric_queries)
    method_queries = ', '.join(method_queries)
    
    event_filter = """('system_order_create_success', 
                    'system_session_start', 
                    'user_product_added', 
                    'system_product_showed', 
                    'system_authentication_success',
                    'system_authentication_showed')
                    """

    stat_query = f"""
    WITH 10000 as limit_const 
    SELECT
        {method_queries}
    FROM (
        SELECT 
            0 as sample_index,
            {metric_queries}
        FROM digital_product_analytics.app_metric_events AS t1
        WHERE t1.event_name IN {event_filter}
        AND toDate(t1.event_datetime) BETWEEN '{config1['start_date']}' AND '{config1['end_date']}'
        --LIMIT limit_const
        
        UNION ALL 
        
        SELECT 
            1 as sample_index,
            {metric_queries}
        FROM digital_product_analytics.app_metric_events AS t1
        WHERE t1.event_name IN {event_filter}
        AND toDate(t1.event_datetime) BETWEEN '{config2['start_date']}' AND '{config2['end_date']}'
        --LIMIT limit_const
    )
    """
    return stat_query

def upload_from_clickhouse(clickhouse_config, mode, metric_dict, config1, config2=None, method_dict=None):
    """
    Динамически формирует и выполняет SQL-запрос к ClickHouse, используя параметры конфигурации и словарь SQL-запросов.

    Аргументы:
        clickhouse_config (dict): Конфигурация для подключения к ClickHouse.
        mode (str): Режим запроса ('data' или 'stat').
        metric_dict (dict): Словарь метрик и их SQL-запросов.
        config1 (dict): Основная конфигурация для параметров запроса.
        config2 (dict, optional): Дополнительная конфигурация для статистических запросов.
        method_dict (dict, optional): Словарь статистических методов.

    Возвращает:
        pandas.DataFrame: Данные, загруженные из ClickHouse.

    Вызывает:
        Exception: Если все попытки выполнения запроса завершились неудачей.
    """
    MAX_ATTEMPTS = 3
    df_results = []

    for attempt in range(MAX_ATTEMPTS):
        try:
            client = get_client(**clickhouse_config)
            query_func = make_query if mode == 'data' else get_stat_query
            query_args = (metric_dict, config1) if mode == 'data' else (metric_dict, config1, config2, method_dict)
            
            query_result = client.query_df(query_func(*query_args))
            df_results.append(query_result)
            break
        except Exception as e:
            print(f"Error executing query for metrics {metric_dict}: {e}. Attempt {attempt + 1} of {MAX_ATTEMPTS}.")
    
    if df_results:
        return pd.concat(df_results, axis=1)
    else:
        raise Exception("Failed to execute any queries.")

# Конфигурация подключения к ClickHouse
clickhouse_config = {
    "host": "172.20.55.10",
    "port": "8122",
    "user": "georgiy_elesin",
    "password": "kkZJ&P992e34Ce",
    "database": "digital_product_analytics"
  }

# Конфигурации периодов для запросов
config1 = {
    # Определяет начальную и конечную даты для первого периода
    'start_date': '2024-07-01',
    'end_date': '2024-07-30'
}

config2 = {
    # Определяет начальную и конечную даты для второго периода
    'start_date': '2024-08-01',
    'end_date': '2024-08-30'
}

# Словарь SQL-запросов метрик
metric_dict = {
    # Содержит категории метрик (order, session, auth, product) и соответствующие SQL-выражения
    'order':
        {
            'gmv_per_day': 'sum(t1.order_value / 100) / uniqExact(toDate(t1.event_datetime)) as gmv_per_day', 
            'gmv_per_restaurant': 'sum(t1.order_value / 100) / uniqExact(t1.restaurant_id) as gmv_per_restaurant',
            'gmv_per_day_restaurant': 'sum(t1.order_value / 100) / (uniqExact(t1.restaurant_id) * uniqExact(toDate(event_datetime))) as gmv_per_day_restaurant',
            'orders_per_day': 'count(t1.event_name) / uniqExact(toDate(t1.event_datetime)) as orders_per_day',
            'orders_per_restaurant': 'count(t1.event_name) / uniqExact(t1.restaurant_id) as orders_per_restaurant',
            'orders_per_day_restaurant': 'count(t1.event_name) / (uniqExact(t1.restaurant_id) * uniqExact(toDate(event_datetime))) as orders_per_day_restaurant',
            'gmv_inc_auth': "sumIf(t1.order_value / 100, t1.authenticated_flg = 'true') / sum(t1.order_value / 100) AS gmv_inc_auth",
            'gmv_inc_multilanguage': "sumIf(t1.order_value / 100, (t1.language NOT IN ('ru', ' ')) OR t1.language IS NULL) / sum(t1.order_value / 100) as gmv_inc_multilanguage",
            'aov': 'avg(t1.order_value / 100) AS aov',
            'aov_1_sec': 'avg(t1.order_value / 100) / avg(t1.session_length) AS aov_1_sec',
            'aup': 'sum(t1.order_value / 100) / sum(t1.product_qnt) AS aup',
            'acs': 'avg(t1.product_qnt) AS acs',
            'asl': 'avg(t1.session_length) as asl',
        },
    'session':
        {
            'gmv_inc_coupon': "sumIf(t1.price / 100, t1.event_name = 'user_product_added' AND t1.coupon_code != 'UNAVAILABLE') / sumIf(t1.price / 100, t1.event_name = 'user_product_added') AS gmv_inc_coupon",
            'gmv_inc_recs': "(sumIf(t1.price, CASE WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'banner') THEN 'Открытие через ссылку в горизонтальном баннере' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'catalog') THEN 'Переход из каталога меню' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'loyalty') THEN 'Открытие из меню лояльности' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'cart_item') THEN 'Тап на добавленную позицию в корзине' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'combo') THEN 'Тап <Изменить состав> в комбо-блюде' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'variant') THEN 'Переключение варианта в карточке блюда' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'coupon_number') THEN 'Успешно введен номер купона' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'recommendation') THEN 'Тап на рекомендацию с виджетом по блюдам в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'change_button') THEN 'Изменил вариант добавленого блюда в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'plus_button') THEN 'Добавил блюдо, тап <+> в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'add_button') THEN 'Добавил блюдо, тап <Добавить> в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'menu:')) THEN 'Добавил блюдо из Набора в меню' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'product_card:')) THEN 'Добавил блюдо из рекомендации в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'cart:')) THEN 'Добавил блюдо из рекомендации в корзине' ELSE '' END IN ('Добавил блюдо из рекомендации в корзине', 'Добавил блюдо из рекомендации в карточке блюда')) / 100) / sum(t1.price / 100) AS gmv_inc_recs",
            'sessions_per_day': "uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_session_start') / uniqExact(toDate(t1.event_datetime)) as session_per_day",
            'session_per_restaurant': "uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_session_start') / uniqExact(t1.restaurant_id) as session_per_restaurant",
            'session_per_day_restaurant': "uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_session_start') / (uniqExact(t1.restaurant_id) * uniqExact(toDate(event_datetime))) as session_per_day_restaurant",
            'cr': "uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_order_create_success') / uniqExact(t1.kiosk_session_id) AS cr",
            'menu_br': "(uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_session_start') - uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_product_showed')) / uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_session_start') AS menu_br",
            'product_br': "(uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_product_showed') - uniqExactIf(t1.kiosk_session_id, t1.event_name = 'user_product_added')) / uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_product_showed') AS product_br",
            'cart_br': "(uniqExactIf(t1.kiosk_session_id, t1.event_name = 'user_product_added') - uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_cart_showed')) / uniqExactIf(t1.kiosk_session_id, t1.event_name = 'user_product_added') AS cart_br",
            'checkout_br': "(uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_cart_showed') - uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_order_create_success')) / uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_cart_showed') AS checkout_br",
            'aspc': "sumIf(t1.product_qnt, t1.event_name = 'user_product_added') / ifNull(countIf(t1.event_name = 'user_product_added'), 0) AS aspc",
            'ctr': "ifNull(countIf(t1.event_name = 'user_product_added'), 0) / ifNull(countIf(t1.event_name = 'system_product_showed'), 0) AS ctr",
            'avps': "ifNull(countIf(t1.event_name = 'system_product_showed'), 0) / uniqExact(t1.kiosk_session_id) AS avps",
            'coupon_rate': "uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_order_create_success' AND ((t1.coupon_code NOT IN ('false', '')) OR t1.coupon_code IS NULL)) / uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_order_create_success') AS coupon_rate"
        },
    'auth': 
        {
            'aov_auth': "avgIf(t1.order_value / 100, t1.authenticated_flg IN ('true')) AS aov_auth",
            'auth_session_rate': "uniqExactIf(t1.kiosk_session_id, t1.authenticated_flg = 'true') / uniqExact(t1.kiosk_session_id) as auth_session_rate",
            'auth_screen_rate': "uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_authentication_success') / uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_authentication_showed') AS auth_screen_rate",
            'auth_aov_diff': "(if(isNaN(avgIf(t1.order_value / 100, t1.authenticated_flg = 'true')), NULL, avgIf(t1.order_value / 100, t1.authenticated_flg = 'true')) - avg(t1.order_value / 100)) / avg(t1.order_value / 100) AS auth_aov_diff",
            'auth_cr': "ifNull(countIf(t1.event_name = 'system_order_create_success' AND t1.authenticated_flg = 'true'), 0) / uniqExactIf(t1.kiosk_session_id, t1.event_name = 'system_authentication_success') AS auth_cr",
           
        },
    'product':
        {
            'aupps': "sum(t1.price / 100) / sum(t1.product_qnt) AS aupps",
            'aup_product': "sumIf(t1.price / 100, CASE WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'banner') THEN 'Открытие через ссылку в горизонтальном баннере' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'catalog') THEN 'Переход из каталога меню' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'loyalty') THEN 'Открытие из меню лояльности' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'cart_item') THEN 'Тап на добавленную позицию в корзине' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'combo') THEN 'Тап <Изменить состав> в комбо-блюде' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'variant') THEN 'Переключение варианта в карточке блюда' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'coupon_number') THEN 'Успешно введен номер купона' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'recommendation') THEN 'Тап на рекомендацию с виджетом по блюдам в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'change_button') THEN 'Изменил вариант добавленого блюда в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'plus_button') THEN 'Добавил блюдо, тап <+> в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'add_button') THEN 'Добавил блюдо, тап <Добавить> в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'menu:')) THEN 'Добавил блюдо из Набора в меню' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'product_card:')) THEN 'Добавил блюдо из рекомендации в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'cart:')) THEN 'Добавил блюдо из рекомендации в корзине' ELSE '' END = 'Добавил блюдо, тап <Добавить> в карточке блюда') / sum(t1.product_qnt) AS aup_product",
            'aup_set': "sumIf(t1.price / 100,CASE WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'banner') THEN 'Открытие через ссылку в горизонтальном баннере' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'catalog') THEN 'Переход из каталога меню' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'loyalty') THEN 'Открытие из меню лояльности' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'cart_item') THEN 'Тап на добавленную позицию в корзине' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'combo') THEN 'Тап <Изменить состав> в комбо-блюде' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'variant') THEN 'Переключение варианта в карточке блюда' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'coupon_number') THEN 'Успешно введен номер купона' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'recommendation') THEN 'Тап на рекомендацию с виджетом по блюдам в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'change_button') THEN 'Изменил вариант добавленого блюда в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'plus_button') THEN 'Добавил блюдо, тап <+> в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'add_button') THEN 'Добавил блюдо, тап <Добавить> в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'menu:')) THEN 'Добавил блюдо из Набора в меню' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'product_card:')) THEN 'Добавил блюдо из рекомендации в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'cart:')) THEN 'Добавил блюдо из рекомендации в корзине' ELSE '' END = 'Добавил блюдо из Набора в меню') / sum(t1.product_qnt) AS aup_set",
            'aup_recs(cart and product)': "sumIf(t1.price / 100, CASE WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'banner') THEN 'Открытие через ссылку в горизонтальном баннере' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'catalog') THEN 'Переход из каталога меню' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'loyalty') THEN 'Открытие из меню лояльности' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'cart_item') THEN 'Тап на добавленную позицию в корзине' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'combo') THEN 'Тап <Изменить состав> в комбо-блюде' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'variant') THEN 'Переключение варианта в карточке блюда' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'coupon_number') THEN 'Успешно введен номер купона' WHEN (t1.event_name = 'system_product_showed' AND t1.source_code = 'recommendation') THEN 'Тап на рекомендацию с виджетом по блюдам в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'change_button') THEN 'Изменил вариант добавленого блюда в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'plus_button') THEN 'Добавил блюдо, тап <+> в корзине' WHEN (t1.event_name = 'user_product_added' AND t1.source_code = 'add_button') THEN 'Добавил блюдо, тап <Добавить> в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'menu:')) THEN 'Добавил блюдо из Набора в меню' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'product_card:')) THEN 'Добавил блюдо из рекомендации в карточке блюда' WHEN (t1.event_name = 'user_product_added' AND startsWith(t1.source_code, 'cart:')) THEN 'Добавил блюдо из рекомендации в корзине' ELSE '' END IN ('Добавил блюдо из рекомендации в карточке блюда', 'Добавил блюдо из рекомендации в корзине')) / sum(t1.product_qnt) AS aup_recs_cart_and_product",
        }
}

# Словарь из случайных величин, из которых мы проектируем метрики для сравнения(см словарь method dict)
stat_dict = {
            'order_value': "if(t1.event_name = 'system_order_create_success', t1.order_value, NULL) as order_value",
            'auth_order_value': "if(t1.event_name = 'system_order_create_success' and t1.authenticated_flg = 'true', t1.order_value, NULL) as auth_order_value",
            'product_qnt': "if(t1.event_name = 'system_order_create_success', t1.product_qnt, NULL) as product_qnt",
            'session_length': "if(t1.event_name = 'system_order_create_success', t1.session_length, NULL) as session_length",
            'order_value_multilanguage': "if(t1.event_name = 'system_order_create_success' and (t1.language NOT IN ('ru', ' ')) OR t1.language IS NULL, t1.order_value / 100, NULL) as order_value_multilanguage",
            'order_flg': "multiIf(t1.event_name = 'system_order_create_success', 1, t1.event_name = 'system_session_start', 0, NULL) as order_flg",
            'auth_order_flg': "multiIf(t1.event_name = 'system_order_create_success' and t1.authenticated_flg = 'true', 1, t1.event_name = 'system_authentication_success' and t1.authenticated_flg = 'true', 0, NULL) as order_flg",
            'add_flg': "multiIf(t1.event_name = 'user_product_added', 1, t1.event_name = 'system_product_showed', 0, NULL) as add_flg",
            'coupon_flg': "multiIf(t1.event_name = 'system_order_create_success' AND t1.coupon_code NOT IN ('false', ''), 1, t1.event_name = 'system_order_create_success' AND t1.coupon_code IN ('false', ''), 0, NULL) as coupon_flg",
            'auth_order_flg': "multiIf(t1.event_name = 'system_order_create_success' and t1.authenticated_flg = 'true', 1, t1.event_name = 'system_session_start', 0, NULL) as auth_order_flg",
            'auth_flg': "multiIf(t1.event_name = 'system_authentication_success', 1, t1.event_name = 'system_authentication_showed', 0, NULL) as auth_flg",
}

# Словарь из методов для сравнений метрик.
method_dict = {
            'gmv_per_day': 'studentTTest(order_value, sample_index) as gmv_per_day',
            'gmv_per_restaurant': 'studentTTest(order_value, sample_index) as gmv_per_restaurant',
            'gmv_per_day_restaurant': 'studentTTest(order_value, sample_index) as gmv_per_day_restaurant',
            'aov': 'studentTTest(order_value, sample_index) as aov',
            'aov_auth': 'studentTTest(auth_order_value, sample_index) as aov_auth',
            'gmv_inc_auth': 'studentTTest(auth_order_value, sample_index) as gmv_inc_auth',
            'acs': 'studentTTest(product_qnt, sample_index) as acs',
            'aupps': 'studentTTest(product_qnt, sample_index) as aupps',
            'asl': 'studentTTest(session_length, sample_index) as asl',
            'gmv_inc_multilanguage': 'studentTTest(order_value, sample_index) as gmv_inc_multilanguage',
            'aup': 'studentTTest(order_value/product_qnt, sample_index) as aup',
            'aov_1_sec': 'studentTTest(order_value/session_length, sample_index) as aov_1_sec',
            'orders_per_day': 'studentTTest(order_flg, sample_index) as orders_per_day',
            'orders_per_restaurant': 'studentTTest(order_flg, sample_index) as orders_per_restaurant',
            'orders_per_day_restaurant': 'studentTTest(order_flg, sample_index) as orders_per_day_restaurant', 
            'cr': "studentTTest(order_flg, sample_index) as cr",
            'auth_cr': "studentTTest(auth_order_flg, sample_index) as auth_cr",
            'ctr': "studentTTest(add_flg, sample_index) as ctr",
            'coupon_rate': "studentTTest(coupon_flg, sample_index) as coupon_rate",
            'auth_session_rate': "studentTTest(auth_order_flg, sample_index) as auth_session_rate",
            'auth_screen_rate': "studentTTest(auth_flg, sample_index) as auth_screen_rate",
}

res = pd.concat([upload_from_clickhouse(clickhouse_config, 'data', metric_dict, config1),
                 upload_from_clickhouse(clickhouse_config, 'data', metric_dict, config2)]
)

# Установка первой строки в качестве индекса
res.set_index(res.T.iloc[0], inplace=True)

# Преобразование и удаление префиксов
res = res.drop(['interval', 'order.interval', 'auth.interval', 'session.interval', 'product.interval'], axis=1).T

# Преобразование и удаление префиксов из индекса
res.index = res.index.str.replace(r'^(order\.|auth\.|session\.|product\.)', '', regex=True)

#res['diff_abs'] = res[res.columns[1]] - res[res.columns[0]]

res['diff_%'] = round(100 * (res[res.columns[1]] - res[res.columns[0]]) / res[res.columns[0]], 1).astype(str) + '%'
res[res.columns[1]] = res[res.columns[1]].apply(lambda x: str(round(100*x, 1)) + '%' if x <= 1 else str(round(x, 2)))
res[res.columns[0]] = res[res.columns[0]].apply(lambda x: str(round(100*x, 1)) + '%' if x <= 1 else str(round(x, 2)))

stat_res = upload_from_clickhouse(clickhouse_config, 'stat', stat_dict, config1, config2, method_dict)
stat_res = stat_res.T.rename(columns={0: 'stat'})
stat_res['stat'] = stat_res['stat'].apply(lambda x: x['p_value'])
stat_res['sign'] = stat_res['stat'].apply(lambda x:  'Да' if x < 0.05 else 'Нет')
final_res = res.join(stat_res, how='left').fillna('-')
final_res.to_excel('C:\ROSTICS-LAB\project\offline_post_test_pipeline\metrics1.xlsx', index=True)

<>:292: SyntaxWarning: invalid escape sequence '\R'
<>:292: SyntaxWarning: invalid escape sequence '\R'
C:\Users\gpe9038\AppData\Local\Temp\ipykernel_32716\2171767938.py:292: SyntaxWarning: invalid escape sequence '\R'
  final_res.to_excel('C:\ROSTICS-LAB\project\offline_post_test_pipeline\metrics1.xlsx', index=True)



        SELECT 
        order.interval as interval,
        *
        FROM 
        (
        SELECT
            CONCAT('2024-07-01', ' - ', '2024-07-30') as interval,
            sum(t1.order_value / 100) / uniqExact(toDate(t1.event_datetime)) as gmv_per_day, 
sum(t1.order_value / 100) / uniqExact(t1.restaurant_id) as gmv_per_restaurant, 
sum(t1.order_value / 100) / (uniqExact(t1.restaurant_id) * uniqExact(toDate(event_datetime))) as gmv_per_day_restaurant, 
count(t1.event_name) / uniqExact(toDate(t1.event_datetime)) as orders_per_day, 
count(t1.event_name) / uniqExact(t1.restaurant_id) as orders_per_restaurant, 
count(t1.event_name) / (uniqExact(t1.restaurant_id) * uniqExact(toDate(event_datetime))) as orders_per_day_restaurant, 
sumIf(t1.order_value / 100, t1.authenticated_flg = 'true') / sum(t1.order_value / 100) AS gmv_inc_auth, 
sumIf(t1.order_value / 100, (t1.language NOT IN ('ru', ' ')) OR t1.language IS NULL) / sum(t1.order_value / 100) as gmv_inc_multilanguage, 
avg(t1.order

In [14]:
final_res

,2024-07-28 - 2024-07-30,2024-08-28 - 2024-08-30,diff_%,stat,sign
gmv_per_day,172574280.77,188667723.91,9.3%,0.0,Да
gmv_per_restaurant,296264.86,323338.0,9.1%,0.0,Да
gmv_per_day_restaurant,148132.43,161669.0,9.1%,0.0,Да
orders_per_day,314014.5,362806.0,15.5%,0.0,Да
orders_per_restaurant,539.08,621.78,15.3%,0.0,Да
orders_per_day_restaurant,269.54,310.89,15.3%,0.0,Да
gmv_inc_auth,2.7%,2.7%,-1.1%,0.0,Да
gmv_inc_multilanguage,0.5%,0.5%,-10.2%,0.0,Да
aov,549.57,520.02,-5.4%,0.0,Да
aov_1_sec,4.06,4.02,-1.1%,0.0,Да
